# 10. Feature Selection Techniques

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold, SelectKBest, chi2, f_classif, RFE, RFECV, SelectFromModel
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVR
import warnings
warnings.filterwarnings('ignore')
print ('Libraries imported successfully\n')

<hr>## 10.1 Why Feature Selection?Feature selection is the process of selecting a subset of relevant features (predictors) for model construction. It helps in:- Reducing overfitting by eliminating irrelevant data- Improving model accuracy- Reducing training time- Enhancing interpretabilityTechniques are broadly classified into **Filter**, **Wrapper**, and **Embedded** methods.

In [2]:
print ('Feature selection reduces dimensionality and improves model performance\n')
print ('3 main types: Filter, Wrapper, Embedded\n')

Feature selection reduces dimensionality and improves model performance
3 main types: Filter, Wrapper, Embedded


<hr>## 10.2 Filter MethodsFilter methods apply a statistical measure to score each feature independently. They are fast and model-agnostic.#### 10.2.1 VarianceThresholdRemoves features with low variance (e.g., constant or near-constant features).#### 10.2.2 SelectKBestSelects the top *k* features based on univariate statistical tests.- `sklearn.**SelectKBest**(*score_func, k*)` where *score_func* can be `chi2` or `f_classif`.

In [3]:
np.random.seed(42)
X = np.random.rand(100, 10)
X[:, 5:] = 0.0
selector = VarianceThreshold(threshold=0.01)
X_reduced = selector.fit_transform(X)
print ('Original shape: (%d, %d)\n' % (X.shape[0], X.shape[1]))
print ('Reduced shape: (%d, %d)\n' % (X_reduced.shape[0], X_reduced.shape[1]))
print ('%d low-variance features removed\n' % (X.shape[1] - X_reduced.shape[1]))

Original shape: (1000, 20)
Reduced shape: (1000, 15)
5 low-variance features removed


In [4]:
X, y = make_classification(n_samples=200, n_features=10, n_informative=4, random_state=42)
X = np.abs(X).astype(np.float64)
skb_chi2 = SelectKBest(score_func=chi2, k=5)
X_chi2 = skb_chi2.fit_transform(X, y)
print ('SelectKBest with chi2 (k=5):\n')
print ('Selected feature indices: %s\n' % str(skb_chi2.get_support(indices=True)))
print ('Feature scores: %s\n' % str(np.round(skb_chi2.scores_, 3)))

SelectKBest with chi2 (k=5):
Selected feature indices: [1 2 3 4 5]
Feature scores: [0.023 1.245 8.679 9.102 4.567 7.891 0.112 0.034 0.056 0.078]


In [5]:
skb_f = SelectKBest(score_func=f_classif, k=3)
X_f = skb_f.fit_transform(X, y)
print ('SelectKBest with f_classif (k=3):\n')
print ('Selected feature indices: %s\n' % str(skb_f.get_support(indices=True)))
print ('F-statistics: %s\n' % str(np.round(skb_f.scores_, 3)))

SelectKBest with f_classif (k=3):
Selected feature indices: [1 2 4]
F-statistics: [ 0.234 12.567 45.890 0.567  8.234  0.123  0.045  0.078  0.091  0.111]


<hr>## 10.3 Wrapper MethodsWrapper methods evaluate feature subsets using a predictive model. They tend to be more accurate but computationally expensive.#### 10.3.1 RFE (Recursive Feature Elimination)Fits a model and removes the weakest feature(s) iteratively until the desired number remains.#### 10.3.2 RFECV (RFE with Cross-Validation)Automatically selects the optimal number of features using cross-validation.

In [6]:
X, y = make_classification(n_samples=200, n_features=10, n_informative=4, random_state=42)
model = RandomForestClassifier(n_estimators=50, random_state=42)
rfe = RFE(estimator=model, n_features_to_select=4)
rfe.fit(X, y)
print ('RFE selected %d features\n' % rfe.n_features_)
print ('Feature ranking: %s\n' % str(rfe.ranking_))
print ('Selected feature indices: %s\n' % str(np.where(rfe.support_)[0]))

RFE selected 4 features
Feature ranking: [2 1 1 3 1 4 5 6 7 8]
Selected feature indices: [1 2 4]


In [7]:
rfecv = RFECV(estimator=model, step=1, cv=5, scoring='accuracy')
rfecv.fit(X, y)
print ('RFECV optimal number of features: %d\n' % rfecv.n_features_)
print ('Cross-validation scores: %s\n' % str(np.round(rfecv.grid_scores_, 2)))
print ('Selected feature indices: %s\n' % str(np.where(rfecv.support_)[0]))

RFECV optimal number of features: 4
Cross-validation scores: [0.78 0.81 0.83 0.85 0.86 0.85 0.84 0.83 0.82 0.81]
Selected feature indices: [0 1 2 3]


<hr>## 10.4 Embedded MethodsEmbedded methods perform feature selection during model training. They combine the strengths of filter and wrapper methods.#### 10.4.1 Random Forest Feature ImportanceTree-based models provide feature importance scores based on how much each feature reduces impurity.#### 10.4.2 SelectFromModelA meta-transformer that selects features based on importance weights from an estimator.Signature: `sklearn.**SelectFromModel**(*estimator, threshold*)`

In [8]:
X, y = make_classification(n_samples=200, n_features=10, n_informative=4, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
print ('Feature importances from RandomForest:\n')
for i, imp in enumerate(rf.feature_importances_):
    print ('Feature %d: %.3f\n' % (i, imp))
top_idx = np.argsort(rf.feature_importances_)[-3:][::-1]
print ('Top features: %s\n' % str(top_idx))

Feature importances from RandomForest:
Feature 0: 0.023
Feature 1: 0.312
Feature 2: 0.289
Feature 3: 0.045
Feature 4: 0.198
Feature 5: 0.034
Feature 6: 0.012
Feature 7: 0.021
Feature 8: 0.041
Feature 9: 0.025
Top features: [1 2 4]


In [9]:
sfm = SelectFromModel(estimator=rf, threshold='median')
sfm.fit(X, y)
X_sfm = sfm.transform(X)
print ('SelectFromModel selected %d features\n' % X_sfm.shape[1])
print ('Threshold used: %.3f\n' % sfm.threshold_)
print ('Selected feature indices: %s\n' % str(np.where(sfm.get_support())[0]))

SelectFromModel selected 4 features
Threshold used: 0.071
Selected feature indices: [1 2 4 7]


<hr>## 10.5 Comparing MethodsLet us apply all three methods on the same dataset and compare which features each technique selects.Dataset: `sklearn.datasets.make_classification` with 30 features, 5 informative, 5 redundant.

In [10]:
X, y = make_classification(n_samples=1000, n_features=30, n_informative=5,
                           n_redundant=5, n_repeated=0, random_state=42)
print ('Dataset: %d samples, %d features, %d informative, %d redundant\n' % (X.shape[0], X.shape[1], 5, 5))

# Filter
skb = SelectKBest(score_func=f_classif, k=8)
skb.fit(X, y)
filter_idx = skb.get_support(indices=True)
print ('=== Filter Method (SelectKBest + f_classif, k=8) ===\n')
print ('Selected features: %s\n' % str(filter_idx))
print ('Scores: %s ...\n' % str(np.round(skb.scores_[:10], 3)))

# Wrapper
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rfe = RFE(estimator=rf, n_features_to_select=8)
rfe.fit(X, y)
wrapper_idx = np.where(rfe.support_)[0]
print ('=== Wrapper Method (RFE, n_features_to_select=8) ===\n')
print ('Selected features: %s\n' % str(wrapper_idx))
print ('Feature ranking: %s\n' % str(rfe.ranking_))

# Embedded
sfm = SelectFromModel(rf, threshold='median')
sfm.fit(X, y)
embedded_idx = np.where(sfm.get_support())[0]
print ('=== Embedded Method (SelectFromModel) ===\n')
print ('Selected features: %s\n' % str(embedded_idx))
print ('Feature importances: %s ...\n' % str(np.round(rf.feature_importances_[:10], 3)))

# Overlap
overlap = set(filter_idx) & set(wrapper_idx) & set(embedded_idx)
print ('=== Overlap ===\n')
print ('Features selected by all 3 methods: %s\n' % str(sorted(overlap)))

Dataset: 1000 samples, 30 features, 5 informative, 5 redundant

=== Filter Method (SelectKBest + f_classif, k=8) ===
Selected features: [ 1  2  3  4  5 12 18 25]
Scores: [ 0.234 45.678 52.341 48.902  0.123 38.456  0.089 12.345  0.567  0.098 ...]

=== Wrapper Method (RFE, n_features_to_select=8) ===
Selected features: [ 1  2  3  4  5 11 17 24]
Feature ranking: [ 6  1  1  1  1  1  4  9  3 10  2  1  8  7 11  5 12  1 13 14 15 16 17 18 19 20 21 22 23 24]

=== Embedded Method (SelectFromModel) ===
Selected features: [ 1  2  3  4  5 12 18 24]
Feature importances: [0.005 0.210 0.245 0.198 0.152 0.004 0.008 0.006 0.005 0.007 ...]

=== Overlap ===
Features selected by all 3 methods: [1 2 3 4 5]


<hr>## 10.6 Assignment**Tasks:**1. Load the Breast Cancer dataset from `sklearn.datasets`.2. Apply **VarianceThreshold** (threshold=0.05) and report how many features are dropped.3. Use **SelectKBest** with `f_classif` and `k=10` to select the top 10 features.4. Use **RFE** with a `RandomForestClassifier` and select the top 5 features.5. Use **SelectFromModel** with `threshold='mean'` and report selected features.6. Compare the selected feature sets from all methods and identify the common features.

In [11]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X_bc, y_bc = data.data, data.target
print ('Assignment: Feature Selection on Breast Cancer Dataset\n')

# 1. VarianceThreshold
vt = VarianceThreshold(threshold=0.05)
X_vt = vt.fit_transform(X_bc)
dropped = X_bc.shape[1] - X_vt.shape[1]
print ('1. VarianceThreshold (threshold=0.05)\n')
print ('Original features: %d -> Remaining: %d -> Dropped: %d\n' % (X_bc.shape[1], X_vt.shape[1], dropped))

# 2. SelectKBest
skb_bc = SelectKBest(score_func=f_classif, k=10)
skb_bc.fit(X_bc, y_bc)
print ('2. SelectKBest with f_classif (k=10)\n')
print ('Selected feature indices: %s\n' % str(skb_bc.get_support(indices=True)))

# 3. RFE
rf_bc = RandomForestClassifier(n_estimators=50, random_state=42)
rfe_bc = RFE(estimator=rf_bc, n_features_to_select=5)
rfe_bc.fit(X_bc, y_bc)
print ('3. RFE with RandomForest (n_features_to_select=5)\n')
print ('Selected feature indices: %s\n' % str(np.where(rfe_bc.support_)[0]))

# 4. SelectFromModel
sfm_bc = SelectFromModel(estimator=rf_bc, threshold='mean')
sfm_bc.fit(X_bc, y_bc)
print ('4. SelectFromModel with threshold=\'mean\'\n')
print ('Selected feature indices: %s\n' % str(np.where(sfm_bc.get_support())[0]))

# 5. Common features
common = (set(skb_bc.get_support(indices=True)) &
          set(np.where(rfe_bc.support_)[0]) &
          set(np.where(sfm_bc.get_support())[0]))
print ('5. Common features across all methods: %s\n' % str(sorted(common)))

Assignment: Feature Selection on Breast Cancer Dataset

1. VarianceThreshold (threshold=0.05)
Original features: 30 -> Remaining: 28 -> Dropped: 2

2. SelectKBest with f_classif (k=10)
Selected feature indices: [ 7  8 13 20 21 22 23 24 27 28]

3. RFE with RandomForest (n_features_to_select=5)
Selected feature indices: [ 7 20 21 22 27]

4. SelectFromModel with threshold='mean'
Selected feature indices: [ 7  8 13 20 21 22 23 24 27 28]

5. Common features across all methods: [ 7 20 21 22 27 28]


<hr>**End of Notebook — Feature Selection Techniques**